# Inference Comparisons and Percentage Reporting

Reproduce three distinct conditional historical populations: baseline mixed-model calibration (40,000), saved-fit CR2 development diagnostics (480), and fresh CR2 validation (26,000). These are not the prospective paired 10,000-input experiment, nor a substitute for it. The diagnostic reuses baseline inputs/fits: a clean reproduction creates the required originals once and then reads their saved RDS objects. No private cached results or report CSVs are required.

All calculations below are disabled. Select a Python 3.10.9 kernel before installation. The reference uses NumPy 1.23.5, SciPy 1.10.0, Node 22.14.0 and R 4.2.2. Exact R package versions are displayed below (including Matrix 1.6-5, lme4 1.1-35.5, lmerTest 3.1-3 and clubSandwich 0.7.0); they must already be installed in the selected R library. Package versions alone do not certify BLAS/compiler parity. Small execution verifies interfaces, not full calibration.

These budget-only reference calculations do not adopt human inference, establish recruitment sufficiency, or validate transfer to the capped model.

In [ ]:
from pathlib import Path
import subprocess, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists() and (p / 'src').is_dir())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT), '-c', str(ROOT / 'configs/native_reference_constraints.txt')])
sys.path.insert(0, str(ROOT / 'src'))


In [ ]:
import json, shlex
RUN = False
FULL = False
RUN_PERCENTAGE = False
FAMILIES = ('baseline', 'diagnostic', 'fresh')
START = 0
LIMIT = 1
OUTPUT = ROOT / 'results/statistical_comparisons'
RSCRIPT = 'Rscript'
NODE = 'node'
environment = json.loads((ROOT / 'src/tutoring_statistics/data/numerical_environment.json').read_text())
print(json.dumps(environment['statistical_reference'], indent=2))
from tutoring_statistics.comparison_design import TOTAL, SCOPE
for family in FAMILIES:
    print(family, TOTAL[family], SCOPE[family])


## Original Populations

Baseline: four views by six scenarios, S1-S4 with 2,000 replicates and S5/S6 with 1,000, preserving all 24 original rep1 inputs. Design seed is 202610190 + 10000 * scenarioIndex + rep; response seed is 302610190 + 10000 * caseIndex + rep. The saved-fit diagnostic is exactly reps1..20 of these same 24 cases. Both use `comparison_cache/production`; keep the same OUTPUT to reuse fits.

Fresh validation: original case indices 1/2/4/7/8/9/10/13/14/16/19/20/22, reps1..2000, design base 402610190 and response base 502610190 with the same stride10000 rule. It uses a separate cache. All populations retain 154 participants per arm and ten rows per person.

Full mode selects these exact original rosters. Small mode selects original ordinals, not a smaller or redrawn scientific population. Full-process rates stay unavailable for incomplete cases. Statistical guard failures remain in planned denominators; missing or technically unresolved work is explicitly separate.

In [ ]:
commands = []
for family in FAMILIES:
    command = [sys.executable, str(ROOT / 'scripts/reproduce_statistical_comparisons.py'), '--family', family, '--stage', 'all', '--execute', '--output', str(OUTPUT), '--rscript', RSCRIPT, '--node', NODE]
    command += ['--full'] if FULL else ['--start', str(START), '--limit', str(LIMIT)]
    commands.append(command)
    print(shlex.join(command))
if RUN:
    for command in commands:
        subprocess.run(command, cwd=ROOT, check=True)


## Independent Readback and Interpretation

The all stage generates native covariates and original Gaussian responses, fits the unchanged model in R, extracts actual RDS evidence independently, verifies exact X/y in R before numeric transport, reconstructs baseline df and CR2/Satterthwaite, checks both tails and Holm4 events, then reduces. `--stage prepare` only generates inputs; `--stage run` fits and checks selected members; `--stage check` reads existing stages and never fits or generates inputs; `--stage aggregate` reduces committed checked events. Completed members resume without draws/fits. Changed source, seed, data or event bindings and incomplete fit attempts are refused, not silently repaired.

Outputs are `baseline/aggregate_full.json`, `diagnostic/aggregate_full.json`, `fresh/aggregate_full.json`, or explicitly named aggregate_partial files. Each contains planned, present, completed, missing and unresolved counts; contrast coverage and both family tail branches retain pointwise Clopper-Pearson intervals. The diagnostic uses corrected count assignment (M=20, never 40), paired transitions, planning diagnostics and per-fit fixed-estimated-W known-DGP SE ratios. This oracle is descriptive, not a usable human estimator.

Negative historical findings are not acceptance targets that overwrite computed output: baseline had 39,999/40,000 usable fits, S4 literal FWER 8.60%-9.65%, coverage 86.90%-88.90%, and assumed-effect family power 5.10%-8.70%. The 480 diagnostic retained seven net coverage decreases. Fresh validation established its pointwise practical criterion in six of thirteen cases; terminal-amount S2 C1-C3 A/B coverage .9365/.9380 remained adverse. Its S4 baseline FWER 8.35%-9.85% is a different population. Later paired10k does not erase these results. This notebook contains no saved outputs and does not claim a newly executed full reproduction.

In [ ]:
if RUN:
    for family in FAMILIES:
        for path in sorted((OUTPUT / family).glob('aggregate_*.json')):
            result = json.loads(path.read_text())
            print(family, path.name, {k: result[k] for k in ('complete', 'expectedTotal', 'completedTotal', 'technicalUnresolved')})
            if family == 'diagnostic':
                print('Negative coverage changes:', result['negativeCoverageChanges'])
            if family == 'fresh':
                print([(c['caseID'], c['practicalTarget']) for c in result['cases']])


## H8/H9 Percentage Software

The source-calling examples below preserve the accepted ordered-L person means and Student test. Effect, SE and CI change to percentage-point units; t, df, p, status and decision do not change. The cancellation example displays a mean of exactly50 while retaining a nonzero centered effect. Testing rounded displayed percentages is diagnostic only. Complete-ten-trial, positive-budget person eligibility and tie handling are unchanged.

The standalone module accepts `--example ordinary|cancellation` or `--input` explicitly labelled synthetic canonical rows. Synthetic raw/native bundles additionally require `--design` and are replayed by the shared normalizer before adaptation. `--design-only` explicitly labels the no-fit percentage route. Human-origin inputs are refused. This is software capability, not new H8/H9 calibration or release of human inference.

In [ ]:
if RUN_PERCENTAGE:
    from tutoring_statistics.comparison_percentage import main as percentage_main
    for example in ('ordinary', 'cancellation'):
        target = OUTPUT / 'percentage' / (example + '.json')
        percentage_main(['--example', example, '--design-only', '--output', str(target)])
        result = json.loads(target.read_text())['percentage']
        for hypothesis in ('H8', 'H9'):
            print(example, hypothesis, result[hypothesis], result['directPercentDiagnostics'][hypothesis]['status'])
